### Problem 001: Unique Paths (LeetCode 62)

### Problem Definition and Constraints
There is an `m x n` grid where you are allowed to move either **down** or **to the right** at any point in time. 
Given the two integers `m` and `n`, return the number of possible unique paths that can be taken from the top-left corner of the grid (`grid[0][0]`) to the bottom-right corner (`grid[m - 1][n - 1]`).

**Examples:**
* **Example 1:**
  * **Input:** `m = 3, n = 6`
  * **Output:** `28`
* **Example 2:**
  * **Input:** `m = 3, n = 2`
  * **Output:** `3`

**Constraints:**
* 1 <= m, n <= 100

### Core Logic: The "Where Did I Come From?" Principle
Instead of trying to trace every possible path from start to finish (which branches exponentially), we flip the question. 

Imagine you are standing in a random room in the middle of this grid. Because you are only allowed to move **Down** or **Right**, there are only *two possible doors* you could have used to enter your current room:
1. The door from the room directly **Above** you.
2. The door from the room directly to your **Left**.

Therefore, the total number of unique ways to reach your current room is simply:
**(Ways to reach the room Above) + (Ways to reach the room to the Left)**

**The Base Cases (The Edges):**
What if you are in the very top row? There is no room above you. You could only have marched straight across from the left. There is only exactly **1** way to reach any room in the top row.
Similarly, for the leftmost column, there is no room to your left. You could only have dropped straight down. There is exactly **1** way to reach any room in the left column.

### Approach 1: 2D Dynamic Programming ($O(m \times n)$ Space)
We create a 2D matrix (a grid of rows and columns). We pre-fill the top row and left column with `1`s. Then, we loop through the remaining empty rooms one by one, adding the value from the room above and the room to the left. 
* **Time Complexity:** $O(m \times n)$ — We visit every room in the grid exactly once.
* **Space Complexity:** $O(m \times n)$ — We store the entire 2D grid in memory.

### Approach 2: Space Optimization ($O(n)$ Space)
If you look closely at the engine `dp[r][c] = dp[r-1][c] + dp[r][c-1]`, you realize that to calculate the current room, we *only* ever look at the current row we are on, and the single row directly above us. The rest of the historical grid above that is completely useless.
Instead of keeping a massive 2D grid, we can just keep a single 1D array that represents the "Row Above." As we calculate our new current row, we just overwrite the old values. 
* **Time Complexity:** $O(m \times n)$
* **Space Complexity:** $O(n)$ — We only store a single row (the width of the grid) in memory.

In [ ]:
class Solution:
    # ---------------------------------------------------------
    # Approach 1: Standard 2D DP Matrix (O(m * n) Space)
    # ---------------------------------------------------------
    def uniquePaths_2D(self, m: int, n: int) -> int:
        # Create an m x n matrix filled with 1s. 
        # Why 1s? This brilliantly handles our base cases automatically!
        # The entire top row (r=0) and left column (c=0) will start with 1,
        # which is exactly what we want because there is only 1 way to reach them.
        dp = [[1] * n for _ in range(m)]
        
        # Start iterating from row 1 and col 1 (skipping the base case edges)
        for r in range(1, m):
            for c in range(1, n):
                # The Core Engine: 
                # Ways to reach here = (Ways to reach Above) + (Ways to reach Left)
                dp[r][c] = dp[r - 1][c] + dp[r][c - 1]
                
        # The bottom-right corner now holds the accumulation of all possible paths
        return dp[m - 1][n - 1]

    # ---------------------------------------------------------
    # Approach 2: 1D Sliding Row (O(n) Space)
    # ---------------------------------------------------------
    def uniquePaths(self, m: int, n: int) -> int:
        # We only keep track of a single row at a time.
        # It starts representing the very top row, so it's all 1s.
        row = [1] * n
        
        # Loop through the remaining rows (from row 1 down to row m-1)
        for r in range(1, m):
            # For each column, calculate the new path combinations
            # We can start at column 1, because the leftmost edge (col 0) is always 1
            for c in range(1, n):
                
                # The Core Engine (Optimized):
                # row[c] currently holds the value from the row ABOVE us.
                # row[c - 1] was just updated in the previous loop iteration, 
                # so it holds the fresh value from our LEFT.
                # We add them together and overwrite our current spot.
                row[c] = row[c] + row[c - 1]
                
        # Once we finish all rows, the very last item in our array is the bottom-right room.
        return row[n - 1]

### Problem 002: Longest Common Subsequence (LeetCode 1143)

### Problem Definition and Constraints
Given two strings `text1` and `text2`, return the length of the longest common subsequence between the two strings if one exists, otherwise return `0`.
A subsequence is a sequence that can be derived from the given sequence by deleting some or no elements without changing the relative order of the remaining characters.
A common subsequence of two strings is a subsequence that exists in both strings.

**Examples:**
* **Example 1:**
  * **Input:** `text1 = "cat", text2 = "crabt"`
  * **Output:** `3`
  * *Explanation:* The longest common subsequence is "cat" which has a length of 3.
* **Example 2:**
  * **Input:** `text1 = "abcd", text2 = "abcd"`
  * **Output:** `4`
* **Example 3:**
  * **Input:** `text1 = "abcd", text2 = "efgh"`
  * **Output:** `0`

**Constraints:**
* 1 <= text1.length, text2.length <= 1000
* `text1` and `text2` consist of only lowercase English characters.

### Core Logic: The "Grid of Decisions" (Diagonal vs. Lateral)
Imagine setting this up as a 2D grid, where `text1` forms the rows and `text2` forms the columns. We add an extra "empty" row at the top and an "empty" column on the left, initialized to `0`, representing comparing against an empty string.

As we evaluate each cell (comparing a character from `text1` against `text2`), we face two scenarios:
1. **The Match (Diagonal Move):** The characters are identical! We found a piece of our common subsequence. We gain `1` point. Because we used both characters, we must look at the best score we had *before* either of these characters were introduced. Physically, this means looking at the cell diagonally up and to the left, and adding `1`.
2. **The Mismatch (Lateral Move):** The characters don't match. We can't gain a point. We have to "delete" a character from one of the strings to keep searching. We look at two options:
   * Delete from `text1` (Look at the cell directly Above).
   * Delete from `text2` (Look at the cell directly to the Left).
   We simply take the maximum score of those two options and carry it forward.

### Approach 1: 2D Dynamic Programming ($O(m \times n)$ Space)
We build an `(m + 1) x (n + 1)` matrix initialized to `0`. We loop through every combination of characters. If they match, we pull from the diagonal + 1. If they don't, we take the max of the top or left cell.
* **Time Complexity:** $O(m \times n)$ — We evaluate every character pairing exactly once.
* **Space Complexity:** $O(m \times n)$ — We store the entire 2D grid.

### Approach 2: Space Optimization ($O(\min(m, n))$ Space)
Just like in *Unique Paths*, calculating the current row only requires data from the *current row* (Left) and the *previous row* (Above / Diagonal). We do not need the entire historical matrix. We can optimize this by only keeping two 1D arrays: one for the previous row and one for the current row we are building.
* **Time Complexity:** $O(m \times n)$
* **Space Complexity:** $O(\min(m, n))$ — We ensure the 1D arrays are sized to match the shorter string to save maximum space.

In [ ]:
class Solution:
    # ---------------------------------------------------------
    # Approach 1: Standard 2D DP Matrix (O(m * n) Space)
    # ---------------------------------------------------------
    def longestCommonSubsequence_2D(self, text1: str, text2: str) -> int:
        m, n = len(text1), len(text2)
        
        # Create an (m+1) x (n+1) grid filled with 0s.
        # The extra row and column handle the base case: 
        # an empty string shares 0 common characters with any other string.
        dp = [[0] * (n + 1) for _ in range(m + 1)]
        
        # Loop through every character in text1 (rows)
        for r in range(1, m + 1):
            # Loop through every character in text2 (cols)
            for c in range(1, n + 1):
                
                # Note: r-1 and c-1 because our strings are 0-indexed, 
                # but our DP array is 1-indexed (shifted by 1 for the empty bases)
                if text1[r - 1] == text2[c - 1]:
                    # Match! Take the diagonal (score before both chars) + 1
                    dp[r][c] = 1 + dp[r - 1][c - 1]
                else:
                    # Mismatch! Take the max of ignoring text1's char (Above) 
                    # OR ignoring text2's char (Left)
                    dp[r][c] = max(dp[r - 1][c], dp[r][c - 1])
                    
        # The bottom-right corner holds the length of the complete LCS
        return dp[m][n]

    # ---------------------------------------------------------
    # Approach 2: 2-Row Space Optimization (O(min(m, n)) Space)
    # ---------------------------------------------------------
    def longestCommonSubsequence(self, text1: str, text2: str) -> int:
        # Optimization: Make text2 the shorter string to minimize our array size
        if len(text1) < len(text2):
            text1, text2 = text2, text1
            
        m, n = len(text1), len(text2)
        
        # We only need two rows: the one above us, and the one we are building
        prev_row = [0] * (n + 1)
        curr_row = [0] * (n + 1)
        
        for r in range(1, m + 1):
            for c in range(1, n + 1):
                if text1[r - 1] == text2[c - 1]:
                    # Diagonal value is in prev_row at the previous column
                    curr_row[c] = 1 + prev_row[c - 1]
                else:
                    # Max of Above (prev_row[c]) and Left (curr_row[c-1])
                    curr_row[c] = max(prev_row[c], curr_row[c - 1])
                    
            # The current row is finished. It now becomes the "previous row" 
            # for the next iteration.
            # We must copy the values (or reassign references carefully)
            prev_row = curr_row[:]
            
        # The final answer lives at the end of the last evaluated row
        return prev_row[n]

### Problem 003: Best Time to Buy and Sell Stock with Cooldown (LeetCode 309)

### Problem Definition and Constraints
You are given an integer array `prices` where `prices[i]` is the price of a stock on the `i`th day.
You may buy and sell one stock multiple times with the following restrictions:
*   After you sell your stock, you cannot buy another one on the next day (i.e., there is a cooldown period of one day).
*   You may only own at most one stock at a time.
*   You may complete as many transactions as you like.
Return the maximum profit you can achieve.

**Examples:**
* **Example 1:**
  * **Input:** `prices = [1,3,4,0,4]`
  * **Output:** `6`
  * *Explanation:* Buy Day 0 (1), Sell Day 1 (3), Profit = 2. Cooldown Day 2. Buy Day 3 (0), Sell Day 4 (4), Profit = 4. Total = 6.
* **Example 2:**
  * **Input:** `prices = [1]`
  * **Output:** `0`

**Constraints:**
* 1 <= prices.length <= 5000
* 0 <= prices[i] <= 1000

### Core Logic: The State Machine (The Three Hotel Rooms)
Instead of a standard 2D grid, this problem introduces a **State Machine**. Imagine you are a trader walking through a calendar day by day. Every night, based on your actions, you must sleep in one of three hotel rooms. You want to track the maximum money you could possibly have in your wallet while sleeping in each room.

1.  **The "Holding" Room (You own a stock):**
    *   *How you got here:* You either slept in this room yesterday and did nothing, OR you were in the "Resting" room yesterday and bought a stock today (subtracting the stock price from your wallet).
2.  **The "Sold" Room (You just sold today):**
    *   *How you got here:* You *must* have been in the "Holding" room yesterday, and you sold your stock today (adding the stock price to your wallet). You are forced to leave this room tomorrow.
3.  **The "Resting" Room (Empty hands, ready to buy):**
    *   *How you got here:* You either slept in this room yesterday and did nothing, OR you slept in the "Sold" room yesterday (serving your mandatory 1-day cooldown) and transitioned here today.

At the end of the timeline, your maximum profit will be the most money in either the "Resting" room or the "Sold" room (you would never end the timeline in the "Holding" room, because you could have just sold it for extra cash).

### Approach 1: Parallel DP Arrays (O(n) Space)
We create three arrays: `hold`, `sold`, and `rest`, where `hold[i]` represents the max profit on day `i` if we end the day in the Holding state. We calculate day `i` by looking at the states from day `i-1`.
* **Time Complexity:** O(n) — We iterate through the prices array exactly once.
* **Space Complexity:** O(n) — We maintain three arrays of size n.

### Approach 2: Sliding Variables (O(1) Space)
Just like in *Unique Paths* and *Longest Common Subsequence*, to calculate today's hotel rooms, we only need to look at *yesterday's* hotel rooms. The rest of the history is useless. We can replace the arrays with three simple variables that update as we walk through the days.
* **Time Complexity:** O(n)
* **Space Complexity:** O(1) — We only track three variables representing our current state.

In [ ]:
from typing import List

class Solution:
    # ---------------------------------------------------------
    # Approach 1: Parallel DP Arrays (O(n) Space)
    # ---------------------------------------------------------
    def maxProfit_array(self, prices: List[int]) -> int:
        if not prices:
            return 0
            
        n = len(prices)
        hold = [0] * n  # Ledger tracking max money if we end the day holding a stock
        sold = [0] * n  # Ledger tracking max money if we end the day having just sold
        rest = [0] * n  # Ledger tracking max money if we end the day doing nothing (cooldown/waiting)
        
        # Base Cases (Day 0)
        hold[0] = -prices[0] # We bought the stock on Day 0, wallet becomes negative to reflect the purchase cost.
        sold[0] = float('-inf') # Impossible to sell on Day 0 because we didn't own a stock before today.
        rest[0] = 0 # Doing nothing on Day 0 leaves our wallet at the starting balance of $0.
        
        for i in range(1, n):
            # To be Holding today: keep the cheapest historical buy (hold[i-1]), 
            # OR use our banked cash (rest[i-1]) to buy at today's price.
            hold[i] = max(hold[i - 1], rest[i - 1] - prices[i])
            
            # To be Sold today: MUST have held a stock yesterday. We sell it at today's price to lock in profit.
            sold[i] = hold[i - 1] + prices[i]
            
            # To be Resting today: keep resting with our current banked cash (rest[i-1]), 
            # OR absorb the money from yesterday's sale (sold[i-1]), serving our 1-day cooldown.
            rest[i] = max(rest[i - 1], sold[i - 1])
            
        # We want the max money without holding a stock at the very end (holding at the end is a waste).
        return max(rest[n - 1], sold[n - 1])

    # ---------------------------------------------------------
    # Approach 2: Sliding Variables (O(1) Space)
    # ---------------------------------------------------------
    def maxProfit(self, prices: List[int]) -> int:
        if not prices:
            return 0
            
        # The Memory (Base Cases for before Day 0)
        # We start with $0. It is impossible to hold or have just sold before the market opens.
        hold = float('-inf') 
        sold = float('-inf')
        rest = 0 
        
        for price in prices:
            # We must remember yesterday's 'sold' value before we overwrite it with today's math,
            # because today's 'rest' calculation requires yesterday's 'sold' value (the cooldown delay).
            prev_sold = sold
            
            # The State Transitions (Moving between the hotel rooms)
            
            # 1. Update Sold: Take the cheapest buy we are holding, and sell it at today's price.
            sold = hold + price
            
            # 2. Update Hold: Keep holding our previous best buy, OR use our resting bank account to buy today's stock.
            hold = max(hold, rest - price)
            
            # 3. Update Rest: Keep our current resting bank account, OR absorb yesterday's sale profit into the bank.
            rest = max(rest, prev_sold)
            
        # Return the max wallet balance from the states where we don't own a stock.
        return max(rest, sold)